# Customer Segmentation using RFM Analysis and K-Means Clustering

This notebook implements a simple, end-to-end customer segmentation pipeline on the **Online Retail II** dataset.
We perform RFM analysis to quantify customer behavior, and then apply K-Means clustering to discover distinct groups.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import datetime

# Enable inline plotting
%matplotlib inline
sns.set_theme(style="whitegrid")

## Step 1: Loading the Dataset

In [ ]:
data_path = '../data/online_retail_II.xlsx'

if not os.path.exists(data_path):
    print(f"Dataset not found at {data_path}. Please download it first!")
else:
    df = pd.read_excel(data_path, sheet_name="Year 2009-2010")
    print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns.")

## Step 2: Data Cleaning

We clean the transaction data by:
1. Removing records missing a `Customer ID`.
2. Keeping only positive quantities and unit prices (removing cancellations and data errors).
3. Computing the total transaction value for each row.

In [ ]:
# Drop rows missing Customer ID
df = df.dropna(subset=['Customer ID'])
df['Customer ID'] = df['Customer ID'].astype(int)

# Filter out cancellations and non-positive prices
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]

# Compute TotalSum
df['TotalSum'] = df['Quantity'] * df['Price']

print(f"Cleaned dataset size: {df.shape}")
df.head()

## Step 3: RFM Metrics Computation

We group by each unique customer and calculate:
- **Recency**: Days between the customer's last purchase and the latest date in the dataset.
- **Frequency**: Number of unique transactions (Invoice counts).
- **Monetary**: Total amount spent by the customer.

In [ ]:
# Define reference date (1 day after the latest purchase)
reference_date = df['InvoiceDate'].max() + datetime.timedelta(days=1)

# Calculate RFM metrics per customer
rfm = df.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (reference_date - x.max()).days,
    'Invoice': 'nunique',
    'TotalSum': 'sum'
})

rfm.rename(columns={
    'InvoiceDate': 'Recency',
    'Invoice': 'Frequency',
    'TotalSum': 'Monetary'
}, inplace=True)

print(f"RFM DataFrame shape: {rfm.shape}")
rfm.head()

## Step 4: Scaling & Outlier Capping

Since K-Means is highly sensitive to outliers, we will cap the variables at the 99th percentile, then scale them using standard scaling.

In [ ]:
# Outlier capping at 99th percentile
rfm_capped = rfm.copy()
for col in ['Recency', 'Frequency', 'Monetary']:
    q_limit = rfm[col].quantile(0.99)
    rfm_capped[col] = np.clip(rfm_capped[col], 0, q_limit)

# Scale metrics
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_capped)
print("RFM metrics capped and standardized.")

## Step 5: Finding Optimal K (Elbow Method)

In [ ]:
wcss = []
k_range = range(1, 11)
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(rfm_scaled)
    wcss.append(kmeans.inertia_)

# Plot
plt.figure(figsize=(8, 5))
plt.plot(k_range, wcss, marker='o', linestyle='--', color='#2c3e50')
plt.title('Elbow Method for Optimal K')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('WCSS (Inertia)')
plt.show()

## Step 6: Customer Clustering with K-Means (K=4)

Let's choose $K=4$ based on the elbow curve, run clustering, and inspect the resulting cluster profiles.

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

# Show profiles
cluster_profiles = rfm.groupby('Cluster').mean()
cluster_counts = rfm.groupby('Cluster').size().to_frame(name='Count')
pd.concat([cluster_profiles, cluster_counts], axis=1)

## Step 7: Visualization

Let's visualize the clusters in a 3D scatter plot.

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
colors = ['#1abc9c', '#3498db', '#e74c3c', '#f1c40f']

for cluster_id in range(4):
    cluster_data = rfm[rfm['Cluster'] == cluster_id]
    ax.scatter(
        cluster_data['Recency'], 
        cluster_data['Frequency'], 
        cluster_data['Monetary'], 
        c=colors[cluster_id], 
        label=f'Cluster {cluster_id}',
        s=40,
        alpha=0.6
    )

ax.set_xlabel('Recency (Days)')
ax.set_ylabel('Frequency (Purchases)')
ax.set_zlabel('Monetary Value ($)')
ax.set_title('Customer Segments Visualization (3D)', fontsize=14, pad=15)
ax.legend()
plt.show()